DSA506 Visual Analytics and Communications  
Michelle Richardson  
Test 1  

**Relief Operations EDA**  

In [170]:
import pandas as pd
import plotly.express as px

The OCHA desk officer must draft a situation report for donor governments by end of day, and she needs your analytical support to back it with evidence.

* Which municipalities are experiencing the largest gaps between supply requested and supply delivered?  
* How does transport mode relate to delivery delays?  
* What is the delivery delay trend during the period?
* Are some supply types under-delivered in general, or to any municipality in particlar?  


In [169]:
df = pd.read_csv('isla_coralina_relief_operations.csv')
df['date'] = pd.to_datetime(df['date'])
date_range = (df['date'].max() - df['date'].min()).days + 1
df.tail()

,delivery_id,distribution_center_id,municipality,date,supply_type,quantity_requested,quantity_delivered,population_at_center,delivery_delay_hours,transport_mode,road_condition,weather_on_delivery_day,number_of_aid_workers,center_capacity
4223,DEL-04224,IC-0146,Rincon del Este,2025-10-05,Shelter Materials,25,6,297,4.5,Helicopter,No,Rain,8,134
4224,DEL-04225,IC-0146,Rincon del Este,2025-10-05,Shelter Materials,46,23,405,3.8,Boat,No,Rain,8,126
4225,DEL-04226,IC-0150,Rincon del Este,2025-10-05,Medical Supplies,14,8,117,1.5,Truck,Yes,Rain,4,120
4226,DEL-04227,IC-0150,Rincon del Este,2025-10-05,Hygiene Kits,45,23,108,2.9,Truck,Yes,Rain,9,105
4227,DEL-04228,IC-0150,Rincon del Este,2025-10-05,Food,92,56,50,2.8,Truck,Yes,Rain,9,100


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4228 entries, 0 to 4227
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   delivery_id              4228 non-null   str    
 1   distribution_center_id   4228 non-null   str    
 2   municipality             4228 non-null   str    
 3   date                     4228 non-null   str    
 4   supply_type              4228 non-null   str    
 5   quantity_requested       4228 non-null   int64  
 6   quantity_delivered       4228 non-null   int64  
 7   population_at_center     4228 non-null   int64  
 8   delivery_delay_hours     4228 non-null   float64
 9   transport_mode           4228 non-null   str    
 10  road_condition           4228 non-null   str    
 11  weather_on_delivery_day  4228 non-null   str    
 12  number_of_aid_workers    4228 non-null   int64  
 13  center_capacity          4228 non-null   int64  
dtypes: float64(1), int64(5), str(8)
mem

The column data types make sense. There are 4228 rows, and no nans indicated in any column.  

In [5]:
df.describe()

,quantity_requested,quantity_delivered,population_at_center,delivery_delay_hours,number_of_aid_workers,center_capacity
count,4228.000000,4228.000000,4228.000000,4228.000000,4228.000000,4228.000000
mean,830.820956,427.050378,514.515137,3.266958,7.563387,904.746925
std,1184.023169,682.560108,479.252418,2.248786,2.940302,871.890175
min,10.000000,5.000000,50.000000,0.000000,2.000000,100.000000
25%,95.000000,42.000000,146.000000,1.500000,6.000000,260.000000
50%,300.000000,142.000000,392.000000,3.000000,7.000000,678.500000
75%,1145.250000,524.000000,706.000000,4.700000,10.000000,1176.250000
max,10856.000000,7620.000000,2785.000000,12.000000,18.000000,5011.000000


In [6]:
print(df.groupby('supply_type')['quantity_requested'].agg(['count', 'mean', 'median', 'max']))

                   count         mean  median    max
supply_type                                         
Food                1185  1157.778903   864.0   6884
Hygiene Kits         527   267.242884   181.0   1744
Medical Supplies     772   127.319948    87.0    836
Shelter Materials    537    66.430168    47.0    431
Water               1207  1545.933720  1095.0  10856


In [7]:
type = df.groupby('supply_type')[['quantity_requested','quantity_delivered']].sum()
type

,quantity_requested,quantity_delivered
supply_type,,
Food,1371968,697227
Hygiene Kits,140837,73937
Medical Supplies,98291,49662
Shelter Materials,35673,18305
Water,1865942,966438


In [8]:
# Look at the site with the maximum request -- is this realistic (and not an outlier)?
max_req = df[df['quantity_requested'] == 10856]
req_liters_pp_DEL_00459 = max_req['quantity_requested'] / max_req['population_at_center']
req_liters_pp_DEL_00459  # This delivery

458    3.898025
dtype: float64

In [9]:
max_req['population_at_center']

458    2785
Name: population_at_center, dtype: int64

In [10]:
# Delivery requests to IC_0027
deliveries_to_IC_0027 = df[df['distribution_center_id'].str.strip()=='IC-0027']
total_IC_0027 = deliveries_to_IC_0027.groupby('supply_type')[['quantity_requested']].sum().reset_index()
total_IC_0027

,supply_type,quantity_requested
0,Food,91349
1,Hygiene Kits,9106
2,Medical Supplies,4206
3,Shelter Materials,1861
4,Water,94813


In [11]:
total_req_liters_pp_DEL_00459 = total_IC_0027[total_IC_0027['supply_type'] == 'Water']['quantity_requested'].values[0] / max_req['population_at_center']
total_req_liters_pp_DEL_00459


458    34.044165
Name: population_at_center, dtype: float64

If we assume liters, the max delivery represents 3.9 liters per person served by facility IC-0027. The total amount of water requested was 94813, so with 2785 at the center, that is 34 liters per person, or about two days' supply according to humanitarian relief standards*. This means it is safe to assume water the water unit is liters. 

The following unit assumptions are made for 'quantity_requested' and 'quantity_delivered':  
Food: Boxes  
Water: Liters   
Hygiene Kits, Medical Supplies: Kit  
Shelter Materials: Tent or tarp    

*Ref: Humanitarian Charter and Minimum Standards in Disaster Response, The Sphere Project, Oxfam Publishing, 2018  
 https://handbook.spherestandards.org/en/sphere/#ch001

Which municipalities are experiencing the largest gaps between supply requested and supply delivered?    

In [140]:
delivery = df.groupby('municipality')[['quantity_requested','quantity_delivered']].sum().reset_index()
delivery['pct_delivered'] = (delivery['quantity_delivered'] / delivery['quantity_requested']).map('{:.1%}'.format)
delivery

,municipality,quantity_requested,quantity_delivered,pct_delivered
0,Bahia Verde,444986,249781,56.1%
1,Costa Sur,864981,399701,46.2%
2,Puerto Nuevo,1366875,748748,54.8%
3,Rincon del Este,401760,178260,44.4%
4,Sierra Alta,434109,229079,52.8%


In [142]:
fig = px.bar(
    delivery,
    x='municipality',
    y=['quantity_requested', 'quantity_delivered'],
    barmode='stack',
    title='Supply Delivery by Municipality: Fulfilled Against Requested',
    #text='pct_delivered',
    labels={
        'municipality': 'Municipality',
        'value': 'Quantity',
        'quantity_requested': 'Requested',
        'quantity_delivered': 'Delivered'
    },
)
fig.update_layout(legend_title_text='Delivery')

fig.show()

All municipalities are receiving about 45-55% of requested supplies. Costa Sur and Rincon del Esta are receiving less than half of their requests.   

How does transport mode map to delivery delay?  

In [167]:
transport = df.groupby(['municipality','transport_mode'], as_index=False)['delivery_delay_hours'].mean()
fig = px.bar(
    transport,
    x='municipality',
    y='delivery_delay_hours',
    color='transport_mode', 
    barmode='group',
    title='Delivery Delay by Municipality for Transport Mode',
    labels={
        'delivery_delay_hours': 'Delay (hours)',
        'transport_mode': 'Mode',
        'municipality': 'Municipality'
    },
)

fig.show()

In [53]:
fig = px.box(
    df,
    x='municipality',
    y='delivery_delay_hours',
    color='transport_mode', 
    title='Delivery Delay for Transport Modes by Municipality',
    labels={
        'delivery_delay_hours': 'Delay (hours)',
        'transport_mode': 'Mode',
        'municipality': 'Municipality'
    },
)

fig.show()

Boat transportation has the highest median delivery delay for all municipalities. The median delay is above 4 hours in all municipalities. Helicopter and truck delivery are close, but vary slightly by municipality. 

In [18]:
df.groupby('municipality')[['center_capacity','population_at_center']].sum()


,center_capacity,population_at_center
municipality,,
Bahia Verde,511632,289686
Costa Sur,936537,528838
Puerto Nuevo,1449756,816206
Rincon del Este,410373,246812
Sierra Alta,516972,293828


In [168]:
bydate_mun = df.groupby(['date','municipality','transport_mode'], as_index=False)['delivery_delay_hours'].mean()
fig = px.line(
    bydate_mun,
    x='date',
    y='delivery_delay_hours',
    color='transport_mode',
    facet_col='municipality',
    facet_col_wrap=2,      # Wrap after 2 columns if there are several modes
    title='Mean Delivery Delay by Municipality and Transport Mode',
    labels={
        'delivery_delay_hours': 'Delay (hours)',
        'date': 'Date',
        'municipality': 'Municipality'
    }
)

fig.show()

In [ ]:
bydate_mun = df.groupby(['date','municipality'], as_index=False)['delivery_delay_hours'].mean()
fig = px.line(
    bydate_mun,
    x='date',
    y='delivery_delay_hours',
    color='municipality',
    title=f'Mean Delivery Delay by Municipality, First {date_range} days',
    labels={
        'delivery_delay_hours': 'Delay (hours)',
        'date': 'Date',
        'municipality': 'Municipality'
    },
    )

fig.show()

Delays seem to share most of the same peaks and drops across municipalities and transport mode. They all seem to be flattening out near or below 4 hours.    

In [158]:
supply_delay = df.groupby(['supply_type'])['delivery_delay_hours'].mean().reset_index()
fig = px.bar(supply_delay,
             x='supply_type',
             y='delivery_delay_hours',
             title='Delivery Delay by Supply Type')
fig.show()

In [173]:
supply_mun = df.groupby(['municipality','supply_type'])['delivery_delay_hours'].mean().reset_index()
fig = px.bar(supply_mun,
             y='supply_type',
             x='delivery_delay_hours',
             color='municipality',
             barmode='stack',
             title='Delivery Delay by Supply Type',
             labels={
                 'supply_type':'Supply Type',
                 'delivery_delay_hours':'Delay (hours)',
                 'municipality':'Municipality'
             })
fig.show()

No significant trends detected in delivery delays by supply type, in general or when broken down by municipality.  

In [177]:
population = df.groupby(['distribution_center_id'])[['center_capacity','population_at_center']].sum().reset_index()
population

,distribution_center_id,center_capacity,population_at_center
0,IC-0001,88682,51225
1,IC-0006,81435,43436
2,IC-0007,86946,46283
3,IC-0008,7650,9509
4,IC-0010,37251,17331
...,...,...,...
67,IC-0137,96756,51714
68,IC-0145,6475,7692
69,IC-0146,6637,9483
70,IC-0149,28750,17340


In [180]:
supply = df.groupby(['distribution_center_id','supply_type'])[['quantity_requested','quantity_delivered']].sum().reset_index()
supply

,distribution_center_id,supply_type,quantity_requested,quantity_delivered
0,IC-0001,Food,36078,20086
1,IC-0001,Hygiene Kits,3090,1801
2,IC-0001,Medical Supplies,1740,991
3,IC-0001,Shelter Materials,840,571
4,IC-0001,Water,39071,24155
...,...,...,...,...
355,IC-0150,Food,4612,2251
356,IC-0150,Hygiene Kits,349,156
357,IC-0150,Medical Supplies,201,103
358,IC-0150,Shelter Materials,126,61
